# 04 — 单模态基线评估

**负责人**: 石韫嘉 | **周次**: W14 | **产出**: 结构化模型 vs 文本模型预测性能对比

## 1. 环境与数据加载

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import compute_metrics, error_summary
from src.evaluation.visualization import (
    plot_predictions, plot_residuals, plot_feature_importance, price_bin_errors
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12})
sns.set_style('whitegrid')
print('Environment ready')

In [ ]:
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

df_results = pd.read_csv(RESULTS_DIR / 'experiment_log.csv')
print(f'Experiment log: {len(df_results)} records')
df_results[['model_name', 'modality', 'status', 'test_rmse', 'test_mae', 'test_r2', 'test_mape']].style \
    .background_gradient(subset=['test_rmse', 'test_mae'], cmap='Reds_r') \
    .background_gradient(subset=['test_r2'], cmap='Greens')

## 2. 加载数据与模型预测

In [ ]:
df = pd.read_csv(DATA_PROCESSED / 'florida_structured.csv')
target_col = 'lastSoldPrice'
drop_cols = [target_col, 'listPrice']
for col in ['_id', '_split', 'sanitized_text', 'clean_text', 'type', 'sub_type', 'zip', 'address', 'description']:
    if col in df.columns:
        drop_cols.append(col)
feature_cols = [c for c in df.columns if c not in drop_cols]

X_struct = np.nan_to_num(df[feature_cols].values.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
y = df[target_col].values.astype(np.float64)
y = np.nan_to_num(y, nan=np.nanmedian(y))
print(f'Structured X: {X_struct.shape}, y: {y.shape}, features: {len(feature_cols)}')

In [ ]:
tfidf = joblib.load(DATA_PROCESSED / 'florida_tfidf_features.pkl').astype(np.float64)
bert_emb = joblib.load(DATA_PROCESSED / 'florida_bert_embeddings.pkl').astype(np.float64)
print(f'TF-IDF: {tfidf.shape}, BERT: {bert_emb.shape}')

In [ ]:
indices = np.arange(len(y))
idx_train, idx_temp = train_test_split(indices, test_size=0.2, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)
X_test = X_struct[idx_test]
y_test = y[idx_test]
X_text_test = {'tfidf': tfidf[idx_test].astype(np.float64), 'bert_embeddings': bert_emb[idx_test].astype(np.float64)}
print(f'Train: {len(idx_train)}, Val: {len(idx_val)}, Test: {len(idx_test)}')

In [ ]:
from src.models.structured_baseline import LinearBaseline, RandomForestBaseline, XGBoostBaseline
from src.models.text_baseline import TFIDFRidgeBaseline, BERTMLPBaseline
from sklearn.preprocessing import StandardScaler

# Mapping: display_name -> (class, modality, model_filename)
model_specs = [
    ('LinearRegression', LinearBaseline, 'structured', 'LinearBaseline.joblib'),
    ('RandomForest', RandomForestBaseline, 'structured', 'RandomForestBaseline.joblib'),
    ('XGBoost', XGBoostBaseline, 'structured', 'XGBoostBaseline.joblib'),
    ('TF-IDF+Ridge', TFIDFRidgeBaseline, 'text', 'TFIDFRidgeBaseline.joblib'),
    ('BERT+MLP', BERTMLPBaseline, 'text', 'BERTMLPBaseline.joblib'),
]

# Get training data for y_scaler repair
y_train = y[idx_train]

predictions = {}
feature_importances = {}

for name, cls, modality, filename in model_specs:
    model_path = MODELS_DIR / filename
    if not model_path.exists():
        print(f'Skip {name}: file {filename} not found')
        continue
    try:
        model = cls.load(str(model_path))
        # Repair: refit _y_scaler if it was lost during serialization
        if hasattr(model, '_y_scaler') and model._y_scaler is None:
            model._y_scaler = StandardScaler()
            model._y_scaler.fit(y_train.reshape(-1, 1))
        if modality == 'structured':
            pred = model.predict(X_test)
            if hasattr(model, 'get_feature_importance'):
                try:
                    feature_importances[name] = model.get_feature_importance()
                except Exception:
                    pass
        else:
            pred = model.predict(None, X_text=X_text_test)
        predictions[name] = pred
        m = compute_metrics(y_test, pred)
        print(f'{name}: RMSE=${m["rmse"]:,.0f}  MAE=${m["mae"]:,.0f}  R2={m["r2"]:.4f}  MAPE={m["mape"]:.1f}%')
    except Exception as e:
        print(f'{name}: error - {e}')

## 3. 性能总览：结构化 vs 文本模型

In [ ]:
baseline = df_results[df_results['modality'].isin(['structured', 'text'])].copy()
table = baseline[['model_name', 'modality', 'test_rmse', 'test_mae', 'test_r2', 'test_mape', 'train_time_sec']].copy()
table.columns = ['Model', 'Modality', 'RMSE', 'MAE', 'R²', 'MAPE(%)', 'TrainTime(s)']
table.sort_values('RMSE').reset_index(drop=True)

In [ ]:
# ============================================================
# FIGURE 1: 4-metric comparison — structured vs text
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
short_names = ['LinearReg', 'RF', 'XGBoost', 'TFIDF+Ridge', 'BERT+MLP']
struct_color = '#4472C4'
text_color = '#ED7D31'
colors = [struct_color]*3 + [text_color]*2

# Fix: use explicit order matching short_names, NOT alphabetical sort
model_order = ['LinearBaseline', 'RandomForestBaseline', 'XGBoostBaseline',
               'TFIDFRidgeBaseline', 'BERTMLPBaseline']
baseline_idx = baseline.set_index('model_name')
ordered = baseline_idx.loc[model_order].reset_index()

for (metric_col, label, fmt, ax) in [
    ('test_rmse', 'RMSE ($)', '${:,.0f}', axes[0,0]),
    ('test_mae', 'MAE ($)', '${:,.0f}', axes[0,1]),
    ('test_r2', 'R²', '{:.4f}', axes[1,0]),
    ('test_mape', 'MAPE (%)', '{:.1f}%', axes[1,1]),
]:
    vals = ordered[metric_col].values
    bars = ax.bar(short_names, vals, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(label, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.015,
                fmt.format(v), ha='center', va='bottom', fontsize=8.5)

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=struct_color, label='Structured'), Patch(color=text_color, label='Text')],
           loc='upper center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, 0.985))
fig.suptitle('Single-Modality Baseline Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_baseline_metrics.png', dpi=200, bbox_inches='tight')
plt.show()

**定量发现**: 结构化模型在所有指标上均显著优于文本模型。LinearReg (RMSE $138,424, R² 0.7436) 是最佳单模态模型，优于 RF 和 XGBoost。BERT+MLP 过拟合严重 (训练 R² 0.75, 测试 R² 0.42)，TF-IDF+Ridge 反而更稳健。

## 4. 预测值 vs 真实值散点图

In [ ]:
# ============================================================
# FIGURE 2: Predicted vs True — all 5 single-modality models
# ============================================================
key_models = ['LinearRegression', 'XGBoost', 'RandomForest', 'TF-IDF+Ridge', 'BERT+MLP']
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flat
for idx, name in enumerate(key_models):
    ax = axes[idx]
    if name not in predictions:
        continue
    pred = predictions[name]
    m = compute_metrics(y_test, pred)
    ax.scatter(y_test, pred, alpha=0.35, s=6, edgecolors='none')
    lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.2)
    ax.set_xlabel('True Price ($)'); ax.set_ylabel('Predicted Price ($)')
    ax.set_title(f'{name}\nRMSE=${m["rmse"]:,.0f} | R²={m["r2"]:.4f} | MAPE={m["mape"]:.1f}%', fontsize=10)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[5].set_visible(False)
fig.suptitle('Predicted vs True — All Single-Modality Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_baseline_pred_vs_true.png', dpi=200, bbox_inches='tight')
plt.show()

**观察**: LinearRegression 散点沿对角线集中度最高（RMSE 最低），证实简单线性模型在充分编码的结构化特征上表现最佳。XGBoost 次之但相近。BERT+MLP 在低价格区间过度预测、高价格区间预测不足；文本模型整体离散度大于结构化模型。

## 5. 残差分布图

In [ ]:
# ============================================================
# FIGURE 3: Residual distributions — all 5 baselines
# ============================================================
all_names = ['LinearRegression', 'RandomForest', 'XGBoost', 'TF-IDF+Ridge', 'BERT+MLP']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flat
for idx, name in enumerate(all_names):
    ax = axes[idx]
    if name not in predictions:
        continue
    res = predictions[name] - y_test
    ax.hist(res, bins=50, color='steelblue', alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.axvline(0, color='red', lw=1.5, ls='--')
    ax.axvline(np.mean(res), color='darkorange', lw=1.5, ls='-', label=f'Mean={np.mean(res):,.0f}')
    ax.set_title(f'{name}\nStd={np.std(res):,.0f} | Skew={np.mean(((res-np.mean(res))/np.std(res))**3):.2f}')
    ax.set_xlabel('Residual ($)'); ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[5].set_visible(False)
fig.suptitle('Residual Distributions — Single-Modality Baselines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_baseline_residuals.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Quantitative residual statistics
res_stats = []
for name in all_names:
    if name in predictions:
        s = error_summary(y_test, predictions[name])
        s['model'] = name
        res_stats.append(s)
pd.DataFrame(res_stats).set_index('model').round(0)

## 6. 特征重要性分析 (树模型)

In [ ]:
# ============================================================
# FIGURE 4: Feature importance — XGBoost & RandomForest (top 20)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for (name, ax) in [('XGBoost', axes[0]), ('RandomForest', axes[1])]:
    if name not in feature_importances:
        continue
    imp = feature_importances[name]
    top_k = min(20, len(imp))
    top_idx = np.argsort(imp)[-top_k:]
    top_idx = top_idx[np.argsort(imp[top_idx])]
    feat_names = [feature_cols[i] for i in top_idx] if len(feature_cols) == len(imp) else [f'F{i}' for i in top_idx]
    cmap = plt.cm.Blues(0.3 + 0.7 * imp[top_idx] / imp[top_idx].max())
    ax.barh(range(len(top_idx)), imp[top_idx], color=cmap, edgecolor='grey', lw=0.3)
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels(feat_names, fontsize=8)
    ax.set_xlabel('Importance (gain)'); ax.set_title(f'{name} — Top {top_k} Features', fontweight='bold')
    ax.grid(axis='x', alpha=0.25)
fig.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_baseline_feature_importance.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. 模态性能差距量化

In [ ]:
# ============================================================
# FIGURE 5: Modality gap — structured vs text
# ============================================================
agg = df_results[df_results['modality'].isin(['structured','text'])].groupby('modality').agg(
    RMSE=('test_rmse','mean'), MAE=('test_mae','mean'), R2=('test_r2','mean'), MAPE=('test_mape','mean')).reset_index()

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for idx, (col, label, fmt) in enumerate([
    ('RMSE', 'Avg RMSE ($)', '${:,.0f}'), ('MAE', 'Avg MAE ($)', '${:,.0f}'),
    ('R2', 'Avg R²', '{:.4f}'), ('MAPE', 'Avg MAPE (%)', '{:.1f}%')]):
    ax = axes[idx]
    vals = [agg[agg['modality']==m][col].values[0] for m in ['structured','text']]
    bars = ax.bar(['Structured','Text'], vals, color=[struct_color, text_color], edgecolor='white', lw=1)
    ax.set_title(label, fontweight='bold')
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
                fmt.format(v), ha='center', va='bottom', fontsize=11, fontweight='bold')
fig.suptitle('Modality Performance Gap: Structured vs Text', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_baseline_modality_gap.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
from src.evaluation.significance_test import improvement_ratio

best_struct_rmse = df_results[df_results['modality']=='structured']['test_rmse'].min()
best_text_rmse = df_results[df_results['modality']=='text']['test_rmse'].min()
print(f'Best Structured RMSE:  ${best_struct_rmse:,.0f}')
print(f'Best Text RMSE:        ${best_text_rmse:,.0f}')
print(f'Absolute gap:           ${best_text_rmse - best_struct_rmse:,.0f}')
print(f'Relative improvement:   {improvement_ratio(best_text_rmse, best_struct_rmse):.1f}%')
print()
print('Conclusion: Structured features dominate prediction power. Text alone cannot match.')
print('Text information may provide incremental gains in fusion models.')

## 8. 小结

| Metric | Best Structured (LinearReg) | Best Text (TF-IDF+Ridge) |
|--------|--------------------------|--------------------------|
| RMSE | $138,424 | $175,621 |
| MAE | $98,184 | $132,936 |
| R² | 0.7436 | 0.5872 |
| MAPE | 404.5% | 501.8% |

**核心发现**: (1) 结构化特征是房价预测的主要信息源，线性回归 (RMSE=$138,424, R²=0.7436) 表现优于更复杂的 RF 和 XGBoost；(2) TF-IDF 文本特征比 BERT 嵌入更稳健；(3) BERT+MLP 存在严重过拟合（训练 R² 0.75, 测试 R² 0.42）；(4) 融合模型有望在结构化基线之上获取文本增量收益。